# 03 — Global Codebase Health

Establish current-state metrics for the build graph. Covers modularity, coupling,
build time decomposition, dependency structure profile, and a cross-team dependency summary.

**Inputs:** Parquet files from `data/processed/` plus outputs of notebooks 01 and 02.

**Output:** `data/results/codebase_health_summary.csv`

In [ ]:
import warnings
from pathlib import Path

import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import networkx as nx
import networkx.algorithms.community as nx_comm
import numpy as np
import pandas as pd
import seaborn as sns
from community import community_louvain

from build_optimiser.config import Config
from build_optimiser.graph import attach_metrics, load_graph, node_centrality

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120

cfg = Config.from_yaml('../config.yaml')
file_df   = pd.read_parquet(cfg.processed_data_dir / 'file_metrics.parquet')
target_df = pd.read_parquet(cfg.processed_data_dir / 'target_metrics.parquet')
edge_df   = pd.read_parquet(cfg.processed_data_dir / 'edge_list.parquet')
G = load_graph(cfg.processed_data_dir / 'edge_list.parquet')
attach_metrics(G, target_df)

# Ensure results dir exists
results_dir = cfg.processed_data_dir.parent / 'results'
results_dir.mkdir(parents=True, exist_ok=True)

print(f'Graph: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges')
print(f'Targets: {len(target_df)}')

## 1. Load Ownership Data

In [ ]:
groups_path    = cfg.processed_data_dir / 'contributor_groups.parquet'
ownership_path = cfg.processed_data_dir / 'target_ownership.parquet'

if groups_path.exists() and ownership_path.exists():
    groups_df    = pd.read_parquet(groups_path)
    ownership_df = pd.read_parquet(ownership_path)
    print(f'Groups    : {len(groups_df)} contributors in {groups_df["group_id"].nunique()} groups')
    print(f'Ownership : {len(ownership_df)} target-group rows')
    # Map each target to its owning group
    target_to_group = (
        ownership_df.loc[ownership_df.groupby('cmake_target')['ownership_normalised'].idxmax(),
                         ['cmake_target', 'group_id']]
        .set_index('cmake_target')['group_id']
        .to_dict()
    )
    has_ownership = True
else:
    print('Ownership data not found — run notebook 02 first. Proceeding without team context.')
    has_ownership = False
    target_to_group = {}
    groups_df = pd.DataFrame(columns=['contributor', 'group_id', 'group_label'])
    ownership_df = pd.DataFrame(columns=['cmake_target', 'group_id', 'ownership_normalised'])

## 2. Modularity Score

In [ ]:
# Convert DAG to undirected for community detection
G_und = G.to_undirected()
# Remove isolates for community detection
G_und_connected = G_und.copy()
G_und_connected.remove_nodes_from(list(nx.isolates(G_und_connected)))

if G_und_connected.number_of_nodes() >= 2:
    # Louvain community detection (Newman's Q)
    louvain_partition = community_louvain.best_partition(G_und_connected, random_state=42)
    louvain_communities = {}
    for node, cid in louvain_partition.items():
        louvain_communities.setdefault(cid, set()).add(node)

    Q_louvain = nx_comm.modularity(G_und_connected, list(louvain_communities.values()))

    print(f'Louvain: {len(louvain_communities)} communities')
    print(f'Newman Q (Louvain)  : {Q_louvain:.4f}')

    # Community sizes
    comm_sizes = sorted([len(v) for v in louvain_communities.values()], reverse=True)
    print(f'Community sizes     : {comm_sizes[:15]}...')
else:
    print(f'Only {G_und_connected.number_of_nodes()} connected node(s) — skipping Louvain.')
    louvain_communities = {0: set(G_und_connected.nodes())}
    Q_louvain = 0.0

In [ ]:
# Ownership partition modularity (if available)
if has_ownership and target_to_group:
    ownership_communities: dict[int, set] = {}
    for node in G_und_connected.nodes():
        gid = target_to_group.get(node, -1)
        ownership_communities.setdefault(gid, set()).add(node)

    Q_ownership = nx_comm.modularity(G_und_connected, list(ownership_communities.values()))
    print(f'Newman Q (ownership partition): {Q_ownership:.4f}')
    print(f'\nInterpretation:')
    print(f'  Q(Louvain)   = {Q_louvain:.4f}  — structural maximum (topological communities)')
    print(f'  Q(ownership) = {Q_ownership:.4f}  — team alignment with graph structure')
    q_gap = Q_louvain - Q_ownership
    print(f'  Gap          = {q_gap:.4f}  ({"teams aligned well" if q_gap < 0.1 else "teams misaligned with code structure"})')
else:
    Q_ownership = None
    print('Skipping ownership modularity (no ownership data)')

## 3. Coupling Metrics

In [ ]:
# For each non-INTERFACE, non-UTILITY target compute coupling metrics
# Edge A -> B: A depends on B
# Afferent coupling (Ca):  number of external targets that depend on this target (in-degree)
# Efferent coupling (Ce):  number of targets this target depends on (out-degree)
# Instability I = Ce / (Ca + Ce)
# Abstractness A = header_files / total_files  (approximation using authored_file_count and unique_headers)
# Distance from main sequence D = |A + I - 1|

coupling_rows = []
for target in G.nodes():
    ca = G.in_degree(target)   # dependants (things that depend on this)
    ce = G.out_degree(target)  # dependencies (things this depends on)
    denom = ca + ce
    instability = ce / denom if denom > 0 else 0.0

    # Abstractness: approximate from file counts in target_df
    row = target_df[target_df['cmake_target'] == target]
    if len(row) == 1:
        total_files = int(row['file_count'].iloc[0] or 0)
        # Unique headers used as proxy for exposed API surface
        unique_hdrs  = int(row['unique_headers_total'].iloc[0] or 0)
        authored_files = int(row['authored_file_count'].iloc[0] or 0)
        abstractness = unique_hdrs / (unique_hdrs + authored_files) if (unique_hdrs + authored_files) > 0 else 0.0
        compile_time = int(row['compile_time_sum_ms'].iloc[0] or 0)
        target_type  = str(row['target_type'].iloc[0])
    else:
        abstractness = 0.0
        compile_time = 0
        target_type  = 'UNKNOWN'

    distance = abs(abstractness + instability - 1.0)
    coupling_rows.append({
        'cmake_target': target,
        'target_type': target_type,
        'Ca': ca,
        'Ce': ce,
        'instability': instability,
        'abstractness': abstractness,
        'distance': distance,
        'compile_time_sum_ms': compile_time,
    })

coupling_df = pd.DataFrame(coupling_rows)
print(f'Coupling metrics for {len(coupling_df)} targets')

# High-distance targets (zone of pain or zone of uselessness)
high_distance = coupling_df[coupling_df['distance'] > 0.4].sort_values('distance', ascending=False)
print(f'Targets with distance D > 0.4: {len(high_distance)}')
print(high_distance[['cmake_target', 'Ca', 'Ce', 'instability', 'abstractness', 'distance']].head(15).to_string(index=False))

In [ ]:
# A-I plane scatter plot
fig, ax = plt.subplots(figsize=(9, 8))

# Main sequence line: A + I = 1
xs = np.linspace(0, 1, 100)
ax.plot(xs, 1 - xs, 'k--', alpha=0.4, linewidth=1.5, label='Main sequence (D=0)')

# Zone labels
ax.text(0.05, 0.05, 'Zone of Pain\n(concrete + stable)', fontsize=8, color='red', alpha=0.6)
ax.text(0.75, 0.9, 'Zone of Uselessness\n(abstract + unstable)', fontsize=8, color='orange', alpha=0.6)

# Scatter by compile time
scatter_df = coupling_df[coupling_df['target_type'].isin(
    ['STATIC_LIBRARY', 'SHARED_LIBRARY', 'MODULE_LIBRARY', 'EXECUTABLE', 'OBJECT_LIBRARY']
)]

sc = ax.scatter(
    scatter_df['instability'], scatter_df['abstractness'],
    c=np.log1p(scatter_df['compile_time_sum_ms']),
    cmap='YlOrRd', alpha=0.7, s=40, edgecolors='grey', linewidths=0.3,
)
plt.colorbar(sc, ax=ax, label='log(1+compile_time_ms)')

# Annotate furthest-from-main-sequence targets
top_outliers = coupling_df.nlargest(8, 'distance')
for _, r in top_outliers.iterrows():
    ax.annotate(
        r['cmake_target'], (r['instability'], r['abstractness']),
        fontsize=6, alpha=0.8,
        xytext=(4, 4), textcoords='offset points',
    )

ax.set_xlim(-0.05, 1.05)
ax.set_ylim(-0.05, 1.05)
ax.set_xlabel('Instability (I = Ce / (Ca+Ce))')
ax.set_ylabel('Abstractness (A)')
ax.set_title('A-I Plane: Coupling Analysis')
ax.legend()
plt.tight_layout()
plt.show()

## 4. Build Time Decomposition

In [ ]:
# Pareto chart: cumulative compile time vs target rank
compile_sorted = (
    target_df[['cmake_target', 'compile_time_sum_ms']]
    .dropna(subset=['compile_time_sum_ms'])
    .sort_values('compile_time_sum_ms', ascending=False)
    .reset_index(drop=True)
)
compile_sorted['cumulative_pct'] = compile_sorted['compile_time_sum_ms'].cumsum() / compile_sorted['compile_time_sum_ms'].sum() * 100

# Find 80% cutoff
cutoff_80 = (compile_sorted['cumulative_pct'] <= 80).sum()

fig, ax1 = plt.subplots(figsize=(14, 5))
ax2 = ax1.twinx()

ax1.bar(compile_sorted.index, compile_sorted['compile_time_sum_ms'] / 1000,
        color='steelblue', alpha=0.7, width=1.0)
ax2.plot(compile_sorted.index, compile_sorted['cumulative_pct'],
         color='crimson', linewidth=2, label='Cumulative %')
ax2.axhline(80, color='crimson', linestyle='--', linewidth=1, alpha=0.6)
ax2.axvline(cutoff_80, color='darkorange', linestyle='--', linewidth=1.5,
            label=f'80% at target #{cutoff_80}')

ax1.set_xlabel('Target rank (sorted by compile time)')
ax1.set_ylabel('Compile time (seconds)')
ax2.set_ylabel('Cumulative %')
ax2.set_ylim(0, 105)
ax1.set_title('Pareto: Compile Time by Target')

lines2, labels2 = ax2.get_legend_handles_labels()
ax2.legend(lines2, labels2, loc='center right')
plt.tight_layout()
plt.show()

total_compile_s = compile_sorted['compile_time_sum_ms'].sum() / 1000
print(f'Total compile time  : {total_compile_s:,.0f} s ({total_compile_s/3600:.2f} h)')
print(f'80% of compile time : top {cutoff_80} targets ({100*cutoff_80/len(compile_sorted):.1f}% of all targets)')

In [ ]:
# Per-team build time share stacked bar chart (if ownership available)
if has_ownership and target_to_group:
    target_df['owning_group'] = target_df['cmake_target'].map(target_to_group)
    team_time = (
        target_df.groupby('owning_group')['compile_time_sum_ms']
        .sum()
        .dropna()
        .sort_values(ascending=False)
    )

    # Add group labels if available
    if 'group_label' in groups_df.columns:
        label_map = groups_df[['group_id', 'group_label']].drop_duplicates().set_index('group_id')['group_label'].to_dict()
        team_time.index = [label_map.get(i, f'group_{i}') for i in team_time.index]

    fig, ax = plt.subplots(figsize=(10, 5))
    colors = sns.color_palette('Set2', len(team_time))
    team_time_s = team_time / 1000
    bars = ax.barh(range(len(team_time_s)), team_time_s.values, color=colors)
    ax.set_yticks(range(len(team_time_s)))
    ax.set_yticklabels(team_time_s.index)
    ax.set_xlabel('Compile time (seconds)')
    ax.set_title('Compile Time by Team')

    for bar, val in zip(bars, team_time_s.values):
        ax.text(bar.get_width() + total_compile_s * 0.005, bar.get_y() + bar.get_height() / 2,
                f'{val:,.0f}s ({100*val/total_compile_s:.1f}%)', va='center', fontsize=8)

    plt.tight_layout()
    plt.show()
else:
    print('Skipping team chart — no ownership data')

## 5. Dependency Structure Profile

In [ ]:
# Topological depth per node (longest path from any root)
# Use longest_path algorithm on each connected component
depth_map: dict[str, int] = {}
try:
    # nx.dag_longest_path_length per node via single-source longest paths
    for node in G.nodes():
        depth_map[node] = 0  # default

    # Compute via topological sort and dynamic programming
    topo = list(nx.topological_sort(G))
    # longest path ending at each node
    lp = {n: 0 for n in G.nodes()}
    for n in reversed(topo):  # process dependencies first
        for pred in G.predecessors(n):
            lp[pred] = max(lp[pred], lp[n] + 1)
    # depth = max over all ancestors
    dp = {n: 0 for n in G.nodes()}
    for n in topo:
        for succ in G.successors(n):
            dp[succ] = max(dp[succ], dp[n] + 1)
    depth_map = dp
except nx.NetworkXUnfeasible:
    print('WARNING: Graph contains cycles — depth computation skipped')

depths = pd.Series(depth_map, name='depth')
depth_series = depths.dropna()

print(f'Max depth  : {depth_series.max()}')
print(f'Mean depth : {depth_series.mean():.1f}')
print(f'Depth distribution:\n{depth_series.value_counts().sort_index().head(20)}')

In [ ]:
# DAG depth histogram overlaid with mean compile time per depth
depth_df = pd.DataFrame({'cmake_target': depth_series.index, 'depth': depth_series.values})
depth_df = depth_df.merge(target_df[['cmake_target', 'compile_time_sum_ms']], on='cmake_target', how='left')
depth_df['compile_time_sum_ms'] = depth_df['compile_time_sum_ms'].fillna(0)

depth_agg = depth_df.groupby('depth').agg(
    count=('cmake_target', 'count'),
    mean_compile_ms=('compile_time_sum_ms', 'mean'),
).reset_index()

fig, ax1 = plt.subplots(figsize=(12, 5))
ax2 = ax1.twinx()

ax1.bar(depth_agg['depth'], depth_agg['count'], color='steelblue', alpha=0.7, label='Target count (width)')
ax2.plot(depth_agg['depth'], depth_agg['mean_compile_ms'] / 1000,
         'o-', color='darkorange', linewidth=2, label='Mean compile time (s)')

ax1.set_xlabel('Topological Depth')
ax1.set_ylabel('Number of Targets (DAG width)')
ax2.set_ylabel('Mean Compile Time per Target (s)')
ax1.set_title('DAG Depth Profile')

h1, l1 = ax1.get_legend_handles_labels()
h2, l2 = ax2.get_legend_handles_labels()
ax1.legend(h1 + h2, l1 + l2)
plt.tight_layout()
plt.show()

In [ ]:
# In-degree and out-degree distributions
in_degrees  = dict(G.in_degree())
out_degrees = dict(G.out_degree())

in_deg_series  = pd.Series(in_degrees, name='in_degree')
out_deg_series = pd.Series(out_degrees, name='out_degree')

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

max_in  = int(in_deg_series.max())
max_out = int(out_deg_series.max())

ax1.hist(in_deg_series, bins=min(50, max_in + 1), color='steelblue', edgecolor='white')
ax1.set_xlabel('In-degree (afferent: # dependants)')
ax1.set_ylabel('Count')
ax1.set_title('In-Degree Distribution')
ax1.set_yscale('log')

ax2.hist(out_deg_series, bins=min(50, max_out + 1), color='darkorange', edgecolor='white')
ax2.set_xlabel('Out-degree (efferent: # dependencies)')
ax2.set_ylabel('Count')
ax2.set_title('Out-Degree Distribution')
ax2.set_yscale('log')

plt.tight_layout()
plt.show()

print(f'In-degree  — mean: {in_deg_series.mean():.2f}, max: {in_deg_series.max()}')
print(f'Out-degree — mean: {out_deg_series.mean():.2f}, max: {out_deg_series.max()}')

# High fan-in targets (depended on by many)
print('\nTop 10 targets by in-degree (fan-in):')
print(in_deg_series.nlargest(10).reset_index().rename(columns={'index': 'cmake_target', 'in_degree': 'in_degree'}).to_string(index=False))

## 6. Cross-Team Dependency Summary

In [ ]:
if has_ownership and target_to_group:
    edge_df_team = edge_df.copy()
    edge_df_team['src_group'] = edge_df_team['source_target'].map(target_to_group)
    edge_df_team['dst_group'] = edge_df_team['dest_target'].map(target_to_group)

    direct_mapped = edge_df_team[
        edge_df_team['is_direct'].eq(True) &
        edge_df_team['src_group'].notna() &
        edge_df_team['dst_group'].notna()
    ]
    cross_team_edges = direct_mapped[direct_mapped['src_group'] != direct_mapped['dst_group']]

    total_direct_mapped = len(direct_mapped)
    total_cross         = len(cross_team_edges)
    cross_fraction      = total_cross / total_direct_mapped if total_direct_mapped > 0 else 0.0

    print(f'Total direct edges (mapped to teams) : {total_direct_mapped}')
    print(f'Cross-team direct edges              : {total_cross}')
    print(f'Cross-team fraction                  : {cross_fraction:.1%}')
else:
    total_direct_mapped = len(edge_df[edge_df['is_direct'].eq(True)])
    total_cross   = 0
    cross_fraction = 0.0
    print('No ownership data — cross-team summary not available')

## 7. Summary Dashboard

In [ ]:
# Betweenness centrality
centrality = node_centrality(G)
centrality_series = pd.Series(centrality)
top_central = centrality_series.nlargest(5)

# Critical path
from build_optimiser.graph import critical_path, critical_path_length
cp = critical_path(G, weight_attr='total_build_time_ms')
cp_length = critical_path_length(G, weight_attr='total_build_time_ms')

print(f'Critical path length : {cp_length / 1000:,.1f} s')
print(f'Critical path nodes  : {len(cp)}')
print('Critical path (first 8):', ' -> '.join(cp[:8]))

In [ ]:
# Assemble summary table
summary = {
    'total_targets': G.number_of_nodes(),
    'total_edges': G.number_of_edges(),
    'total_direct_edges': int(edge_df['is_direct'].eq(True).sum()),
    'total_compile_time_s': round(target_df['compile_time_sum_ms'].sum() / 1000, 1),
    'total_build_time_s': round(target_df['total_build_time_ms'].sum() / 1000, 1),
    'critical_path_s': round(cp_length / 1000, 1),
    'critical_path_nodes': len(cp),
    'dag_max_depth': int(depth_series.max()),
    'dag_mean_depth': round(depth_series.mean(), 2),
    'max_in_degree': int(in_deg_series.max()),
    'max_out_degree': int(out_deg_series.max()),
    'mean_in_degree': round(float(in_deg_series.mean()), 3),
    'mean_out_degree': round(float(out_deg_series.mean()), 3),
    'modularity_louvain': round(Q_louvain, 4),
    'modularity_ownership': round(Q_ownership, 4) if Q_ownership is not None else None,
    'cross_team_edge_count': total_cross,
    'cross_team_edge_fraction': round(cross_fraction, 4),
    'pareto_80pct_target_count': cutoff_80,
    'pareto_80pct_target_fraction': round(cutoff_80 / max(len(compile_sorted), 1), 4),
    'high_distance_target_count': len(high_distance),
    'targets_with_bus_factor_1': None,  # populated below if data available
    'top_central_target': top_central.index[0] if len(top_central) > 0 else None,
}

# Load bus factor if available
try:
    git_detail = pd.read_csv(cfg.raw_data_dir / 'git_history_detail.csv')
    from build_optimiser.contributors import compute_bus_factor
    if has_ownership:
        file_to_target = dict(zip(file_df['source_file'], file_df['cmake_target']))
        git_detail['cmake_target'] = git_detail['source_file'].map(file_to_target)
        git_detail_mapped = git_detail.dropna(subset=['cmake_target'])
        bus_df = compute_bus_factor(git_detail_mapped, groups_df, recent_months=3)
        low_bus_count = bus_df[bus_df['bus_factor'] <= 1]['cmake_target'].nunique()
        summary['targets_with_bus_factor_1'] = low_bus_count
except Exception as e:
    print(f'Bus factor summary skipped: {e}')

summary_df = pd.DataFrame(list(summary.items()), columns=['metric', 'value'])
print(summary_df.to_string(index=False))

In [ ]:
# Visualise summary as a single-page dashboard
key_metrics = [
    ('Targets', summary['total_targets']),
    ('Direct edges', summary['total_direct_edges']),
    ('Total compile (h)', round(summary['total_compile_time_s'] / 3600, 2)),
    ('Critical path (s)', summary['critical_path_s']),
    ('Max DAG depth', summary['dag_max_depth']),
    ('Louvain Q', summary['modularity_louvain']),
    ('Ownership Q', summary['modularity_ownership'] if summary['modularity_ownership'] is not None else 'N/A'),
    ('Cross-team edges', f"{summary['cross_team_edge_count']} ({summary['cross_team_edge_fraction']:.1%})"),
    ('80% Pareto targets', f"{summary['pareto_80pct_target_count']} ({summary['pareto_80pct_target_fraction']:.1%})"),
    ('High-distance targets', summary['high_distance_target_count']),
]

fig, ax = plt.subplots(figsize=(10, 5))
ax.axis('off')
table_data = [[str(v)] for _, v in key_metrics]
row_labels  = [m for m, _ in key_metrics]
table = ax.table(
    cellText=table_data,
    rowLabels=row_labels,
    colLabels=['Value'],
    loc='center',
    cellLoc='left',
)
table.auto_set_font_size(False)
table.set_fontsize(11)
table.scale(1.6, 1.8)
ax.set_title('Codebase Health Summary', fontsize=14, pad=20)
plt.tight_layout()
plt.show()

In [ ]:
# Save summary to CSV
output_path = results_dir / 'codebase_health_summary.csv'
summary_df.to_csv(output_path, index=False)
print(f'Summary written to {output_path}')

# Also save coupling_df for downstream notebooks
coupling_path = cfg.processed_data_dir / 'coupling_metrics.parquet'
import pyarrow as pa
import pyarrow.parquet as pq
pq.write_table(
    pa.Table.from_pandas(coupling_df, preserve_index=False),
    coupling_path,
    compression='snappy',
)
print(f'Coupling metrics written to {coupling_path}')
print('Done.')